# Camera Discovery Harvest Architecture Test Notebook

This Google Colab notebook tests the `camera-discovery harvest-urls` CLI workflow and the harvest handoff into `camera-discovery run`. Harvest mode is extraction-only: it bypasses target resolution, geocoding, validation, trust policy, scope enforcement, LLM review, GeoJSON, map output, `cameras.md`, and review ZIP generation.

The expanded harvester has two parallel extraction lanes: structured camera/feed records from public JSON/API/GeoJSON/ArcGIS-style endpoints, and raw camera/media URL extraction from HTML, JavaScript, network capture, direct seeds, linked endpoints, and escaped/encoded text. Structured records preserve source-provided fields such as coordinates, orientation/direction, timestamps, `inService`, image descriptions, update frequencies, and grouped media assets. These values are source-provided only; they are not validated, trusted, geocoded, or scope-filtered in harvest mode.


Use `--write-intermediate-records` when you need debug/analysis files for the raw block-policy-filtered records, deduped unique records, and media-filtered records before the final `--max-urls` cap. These files can be large, so the flag is opt-in.


In [ ]:
!git clone -b "main" "https://github.com/dshipley71/camera-discovery.git"


In [ ]:
%cd camera-discovery

In [ ]:
# Install the package in editable mode.
%pip install -e .
# %pip install -e .[playwright] --no-build-isolation
# !python -m playwright install chromium
%pip install -e .[cloakbrowser] --no-build-isolation


In [ ]:
# Verify CLI registration and refactored public imports.
!camera-discovery --help
!camera-discovery run --help
!camera-discovery harvest-urls --help

import camera_discovery.cli
import camera_discovery.extraction.media
import camera_discovery.harvest.media_filter
print("refactored camera-discovery CLI/helper imports OK")


## Harvest all supported media types

This command collects raw direct media/camera URLs across supported media categories and writes plain URL and metadata outputs. It does not validate streams or write inventory artifacts.


In [ ]:
%%time
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california \
  --max-urls 5000 \
  --discovery-mode both \
  --enable-browser-capture


## Harvest only HLS / `.m3u8` URLs

This run uses `--max-urls 0` so the final HLS output is uncapped, writes intermediate records for inspection, and records an HLS-only handoff that `run --harvest-input --harvest-input-mode handoff-only` loads as filtered media records by default. Handoff-only disables native discovery; use `--harvest-input-mode seed` only when you intentionally want to combine the handoff with normal discovery.


In [ ]:
%%time
!rm -rf runs/harvest-california-hls
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california-hls \
  --discovery-mode both \
  --max-search-queries 40 \
  --max-search-results-per-query 50 \
  --max-source-rows 5000 \
  --max-pages-per-source 25 \
  --max-urls 0 \
  --media .m3u8 \
  --browser-backend cloakbrowser \
  --write-intermediate-records \
  --progress-style plain


## Inspect optional intermediate harvest records

The HLS harvest command above uses `--write-intermediate-records`, which writes optional debug/analysis JSONL files for the harvest reduction pipeline. `raw_media_records.jsonl` contains block-policy-filtered records before deduplication, `unique_media_records.jsonl` contains deduped records before media filtering, and `media_filtered_records.jsonl` contains records matching the requested media filter before the final `--max-urls` cap.


In [ ]:
from pathlib import Path
import json

HLS_HARVEST_DIR = Path('runs/harvest-california-hls')
intermediate_paths = {
    'raw_media_records.jsonl': HLS_HARVEST_DIR / 'raw_media_records.jsonl',
    'unique_media_records.jsonl': HLS_HARVEST_DIR / 'unique_media_records.jsonl',
    'media_filtered_records.jsonl': HLS_HARVEST_DIR / 'media_filtered_records.jsonl',
    'image_filtered_records.jsonl': HLS_HARVEST_DIR / 'image_filtered_records.jsonl',
    'logs/intermediate_records_summary.json': HLS_HARVEST_DIR / 'logs' / 'intermediate_records_summary.json',
}
for label, path in intermediate_paths.items():
    print(f'{label}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}')

summary_path = HLS_HARVEST_DIR / 'harvest_summary.json'
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    print('intermediate_records_written:', summary.get('intermediate_records_written'))
    print('intermediate_record_counts:', summary.get('intermediate_record_counts'))
    print('intermediate_record_files:', summary.get('intermediate_record_files'))

preview_path = intermediate_paths['media_filtered_records.jsonl']
if preview_path.exists():
    print('First 3 media-filtered records:')
    with preview_path.open(encoding='utf-8') as f:
        for idx, line in enumerate(f):
            if idx >= 3:
                break
            row = json.loads(line)
            print({k: row.get(k) for k in ['url', 'media_type', 'camera_record_id', 'asset_role', 'source_url', 'source_endpoint_url']})


handoff_path = HLS_HARVEST_DIR / 'harvest_handoff.json'
if handoff_path.exists():
    handoff = json.loads(handoff_path.read_text(encoding='utf-8'))
    print('handoff schema:', handoff.get('schema_version'))
    print('handoff media_filter:', handoff.get('media_filter'))
    print('handoff default scope:', handoff.get('handoff_default_scope'))
    print('handoff files:', handoff.get('files'))


## Harvest a media mix: HLS, images, and generic stream URLs


In [ ]:
%%time
!camera-discovery harvest-urls "California traffic cameras" \
  --output-dir runs/harvest-california-media-mix \
  --max-urls 5000 \
  --media hls,image,stream


## Inspect output paths, summary counts, and structured harvest files

The summary includes counts for structured camera records, grouped media assets, discovered endpoints, records with collected coordinates, orientation, `inService`, update frequencies, and date/time metadata. The CSV/JSONL outputs include top-level fields such as `camera_record_id`, `asset_id`, `asset_role`, `lat`, `lon`, `coordinate_source`, `direction`, `bearing`, `heading`, `orientation`, `in_service`, `date`, `time`, `timestamp`, `last_updated`, `last_refresh`, `image_description`, `current_image_update_frequency`, and `reference_image_update_frequency` when those values were present in source metadata.


In [ ]:
from pathlib import Path
import csv
import json

HARVEST_DIR = Path('runs/harvest-california')
paths = {
    'camera_urls.txt': HARVEST_DIR / 'camera_urls.txt',
    'camera_urls.csv': HARVEST_DIR / 'camera_urls.csv',
    'camera_urls.jsonl': HARVEST_DIR / 'camera_urls.jsonl',
    'camera_records.jsonl': HARVEST_DIR / 'camera_records.jsonl',
    'camera_media_assets.jsonl': HARVEST_DIR / 'camera_media_assets.jsonl',
    'discovered_endpoints.jsonl': HARVEST_DIR / 'discovered_endpoints.jsonl',
    'harvest_camera_inventory.jsonl': HARVEST_DIR / 'harvest_camera_inventory.jsonl',
    'harvest_handoff.json': HARVEST_DIR / 'harvest_handoff.json',
    'harvest_summary.json': HARVEST_DIR / 'harvest_summary.json',
    'source_rows.jsonl': HARVEST_DIR / 'source_rows.jsonl',
    'raw_media_records.jsonl': HARVEST_DIR / 'raw_media_records.jsonl',
    'unique_media_records.jsonl': HARVEST_DIR / 'unique_media_records.jsonl',
    'media_filtered_records.jsonl': HARVEST_DIR / 'media_filtered_records.jsonl',
}
for label, path in paths.items():
    print(f'{label}: {path} exists={path.exists()} size={path.stat().st_size if path.exists() else 0}')

summary_path = paths['harvest_summary.json']
if summary_path.exists():
    summary = json.loads(summary_path.read_text(encoding='utf-8'))
    for key in [
        'raw_records', 'unique_urls', 'media_filtered_urls', 'written_urls',
        'structured_camera_records', 'media_assets', 'endpoints_discovered',
        'records_with_coordinates', 'records_with_orientation', 'records_with_in_service',
        'records_with_datetime', 'records_with_update_frequency',
        'records_with_streaming_video', 'records_with_current_image', 'records_with_reference_image',
        'intermediate_records_written',
    ]:
        print(f'{key}:', summary.get(key))
    print('by_media_type:', summary.get('by_media_type'))
    print('intermediate_record_counts:', summary.get('intermediate_record_counts'))
    print('intermediate_record_files:', summary.get('intermediate_record_files'))
    print('outputs:', summary.get('outputs'))

    print('\nSource row provenance:')
    source_rows_summary = summary.get('source_rows') or {}
    for key in [
        'discovery_mode', 'sources_file', 'sources_file_exists', 'sources_file_loaded', 'sources_file_used',
        'directory_requested', 'blind_requested', 'direct_requested',
        'directory_sources_configured', 'directory_sources_enabled', 'blocked_patterns_configured',
        'generated_rows', 'selected_rows_before_budget', 'selected_rows',
        'selected_directory_rows', 'selected_blind_rows', 'selected_direct_rows',
        'blocked_source_rows', 'max_source_rows', 'max_source_rows_applied',
    ]:
        print(f'{key}:', source_rows_summary.get(key))
    print('generated_by_provider:', source_rows_summary.get('generated_by_provider'))
    print('selected_by_provider:', source_rows_summary.get('selected_by_provider'))
    print('selected_by_kind:', source_rows_summary.get('selected_by_kind'))

    # Cross-check source_rows.jsonl directly so it is obvious whether SOURCES.md/directory rows were present.
    source_rows_path = paths['source_rows.jsonl']
    if source_rows_path.exists():
        from collections import Counter
        source_rows = [json.loads(line) for line in source_rows_path.read_text(encoding='utf-8').splitlines() if line.strip()]
        provider_counts = Counter(row.get('source_provider') or 'unknown' for row in source_rows)
        kind_counts = Counter(row.get('source_kind') or 'unknown' for row in source_rows)
        print('source_rows.jsonl provider counts:', dict(provider_counts))
        print('source_rows.jsonl kind counts:', dict(kind_counts))
        print('first directory/SOURCES.md rows:')
        shown = 0
        for row in source_rows:
            if row.get('source_provider') == 'directory':
                print({k: row.get(k) for k in ['source_name', 'source_kind', 'url', 'source_scope_hint']})
                shown += 1
                if shown >= 5:
                    break
        if shown == 0:
            print('No directory rows found in source_rows.jsonl.')

# Preview flattened URL metadata columns.
csv_path = paths['camera_urls.csv']
if csv_path.exists() and csv_path.stat().st_size:
    with csv_path.open(newline='', encoding='utf-8') as f:
        reader = csv.DictReader(f)
        for idx, row in enumerate(reader):
            if idx >= 5:
                break
            print({k: row.get(k) for k in [
                'media_url', 'media_type', 'asset_role', 'camera_record_id', 'camera_id',
                'lat', 'lon', 'direction', 'bearing', 'heading', 'in_service',
                'timestamp', 'last_updated', 'image_description',
                'current_image_update_frequency', 'reference_image_update_frequency',
                'source_endpoint_url', 'json_record_path'
            ]})


## Print the first harvested URLs


In [ ]:
N = 25
url_path = Path('runs/harvest-california/camera_urls.txt')
if url_path.exists():
    for idx, line in enumerate(url_path.read_text().splitlines()[:N], start=1):
        print(f'{idx:03d}: {line}')
else:
    print('No camera_urls.txt file found yet.')


## Preview structured camera records, media assets, endpoints, and handoff manifest

These files are the structured harvest artifacts. `camera_records.jsonl` preserves full source camera objects; `camera_media_assets.jsonl` groups direct media URLs by camera; `discovered_endpoints.jsonl` catalogs JSON/API/feed endpoints; `harvest_camera_inventory.jsonl` and `harvest_handoff.json` are used to seed the normal run workflow.


In [ ]:
def preview_jsonl(path: Path, limit: int = 3):
    print(f"--- {path} ---")
    if not path.exists():
        print("missing")
        return
    for idx, line in enumerate(path.read_text(encoding="utf-8").splitlines()[:limit], start=1):
        try:
            obj = json.loads(line)
            print(idx, json.dumps({k: obj.get(k) for k in list(obj)[:12]}, indent=2)[:2000])
        except Exception:
            print(idx, line[:500])

for file_name in [
    "camera_records.jsonl",
    "camera_media_assets.jsonl",
    "discovered_endpoints.jsonl",
    "harvest_camera_inventory.jsonl",
]:
    preview_jsonl(HARVEST_DIR / file_name)

handoff_path = HARVEST_DIR / "harvest_handoff.json"
if handoff_path.exists():
    print("--- harvest_handoff.json ---")
    print(json.dumps(json.loads(handoff_path.read_text(encoding="utf-8")), indent=2)[:4000])


## Use HLS harvest handoff as normal run input

This command demonstrates the HLS handoff path. The handoff is source-provided and unvalidated, but media-filtered handoffs now default to the filtered final media records. An HLS-only harvest should seed HLS candidates only, not the broader structured inventory of image snapshots unless explicitly implemented in the future. Normal `camera-discovery run` still performs target resolution, deterministic scope gating, validation/trust/output behavior according to configuration.


In [ ]:
import os

RUN_PROFILE = "balanced"
if RUN_PROFILE not in {"fast", "balanced", "full"}:
    raise ValueError(f"Invalid CAMERA_DISCOVERY_PROFILE={RUN_PROFILE!r}; expected fast, balanced, or full")

# User-controlled query. Edit this directly or set CAMERA_DISCOVERY_QUERY in the environment.
# Camera-type terms such as "traffic cameras" are camera intent, not target geography.
# USER_QUERY = os.environ.get("CAMERA_DISCOVERY_QUERY", "Get me all traffic cameras from California")

# Other valid examples:
# USER_QUERY = "Get me all cameras from Greenville, Texas"
# USER_QUERY = "Get me all cameras from London, England and New York, New York"

# Notebook run controls. These are used by the notebook wrapper, not the application config.
OUTPUT_DIR = "runs/notebook-live-test"
CLEAN_OUTPUT_DIR = "true"

# LLM provider/model settings.
# This application requires a real LLM provider. Each application section can use
# a different model. Edit the variables below, or set the matching environment
# variables before running this cell.
LLM_PROVIDER = "ollama-cloud"

# Defaults are intentionally independent so users can test different models per stage.
DEFAULT_MAIN_MODEL = "gemma4:31b-cloud"
DEFAULT_TARGET_INTENT_MODEL = "gemma3:12b-cloud"
DEFAULT_TARGET_INTENT_FALLBACK_MODEL = "gemma3:12b-cloud"
DEFAULT_GEOCODER_REFEREE_MODEL = "gemma4:31b-cloud"
DEFAULT_LOCATION_INFERENCE_MODEL = "gemma4:31b-cloud"
DEFAULT_CANDIDATE_REVIEW_MODEL = "gemma3:12b-cloud"

# For example, after confirming availability in your provider account, you can use:
# DEFAULT_TARGET_INTENT_MODEL = "qwen3.5:4b"           # fast extraction
# DEFAULT_GEOCODER_REFEREE_MODEL = "gemma4:31b-cloud" # stronger semantic ranking
# DEFAULT_CANDIDATE_REVIEW_MODEL = "qwen3.5:4b"       # faster batch review

MAIN_MODEL = DEFAULT_MAIN_MODEL
TARGET_INTENT_MODEL = DEFAULT_TARGET_INTENT_MODEL
TARGET_INTENT_FALLBACK_MODEL = DEFAULT_TARGET_INTENT_FALLBACK_MODEL
TARGET_INTENT_TIMEOUT = "30"
TARGET_INTENT_ATTEMPTS = "1"
GEOCODER_REFEREE_MODEL = DEFAULT_GEOCODER_REFEREE_MODEL
LOCATION_INFERENCE_MODEL = DEFAULT_LOCATION_INFERENCE_MODEL
LOCATION_INFERENCE_TIMEOUT = "45"
ENABLE_LLM_LOCATION_INFERENCE = "true"
LOCATION_INFERENCE_MIN_CONFIDENCE = "0.70"
CANDIDATE_REVIEW_MODEL = DEFAULT_CANDIDATE_REVIEW_MODEL
CANDIDATE_REVIEW_TIMEOUT = "60"
CANDIDATE_REVIEW_BATCH_SIZE = "8"

# Discovery/candidate budgets. Keep these aligned with active RunConfig fields.
# The notebook does not expose a separate total-candidate knob; the source
# total cap is derived below from these visible per-type candidate budgets.
MAX_SEARCH_QUERIES = "10"
MAX_SEARCH_RESULTS_PER_QUERY = "8"
MAX_PAGES = "60"
MAX_HLS_CANDIDATES = "5000"
MAX_IMAGE_SNAPSHOT_CANDIDATES = "0"
MAX_DIRECTORY_PAGES = "10"
MAX_STRUCTURED_ENDPOINTS_PER_PAGE = "25"

# Browser-capture routing controls. Static extraction still runs first; these
# caps bound the second-stage Playwright capture used for dynamic camera pages.
ENABLE_BROWSER_CAPTURE = "true"
BROWSER_BACKEND = "cloakbrowser"
BROWSER_CAPTURE_TIMEOUT_MS = "15000"
BROWSER_CAPTURE_MIN_SCORE = "3"
MAX_BROWSER_CAPTURE_PAGES = "20"
MAX_BROWSER_CAPTURE_PAGES_BLIND = "6"
MAX_BROWSER_CAPTURE_PAGES_DIRECTORY = "12"
MAX_BROWSER_CAPTURE_PAGES_PER_HOST = "3"
BROWSER_CAPTURE_SETTLE_MS = "1000"
BROWSER_CAPTURE_SCROLL = "false"
MAX_BROWSER_JSON_ENDPOINTS_PER_PAGE = "10"
MAX_BROWSER_NETWORK_EVENTS_LOGGED_PER_PAGE = "50"
IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS = "2.0"


def _nonnegative_int_setting(name: str, value: str) -> int:
    try:
        parsed = int(value)
    except ValueError as exc:
        raise ValueError(f"{name} must be an integer, got {value!r}") from exc
    if parsed < 0:
        raise ValueError(f"{name} must be non-negative, got {parsed}")
    return parsed


# Source RunConfig still reads a total candidate cap env var. For notebook runs,
# derive that cap from HLS budget + image-snapshot budget. Do not set the
# deprecated CAMERA_DISCOVERY_MAX_STREAMS compatibility cap from the notebook.
_TOTAL_CANDIDATE_CAP_ENV = "CAMERA_DISCOVERY_" + "MAX_TOTAL_" + "CANDIDATES"
_SOURCE_TOTAL_CANDIDATE_CAP = (
    _nonnegative_int_setting("CAMERA_DISCOVERY_MAX_HLS_CANDIDATES", MAX_HLS_CANDIDATES)
    + _nonnegative_int_setting("CAMERA_DISCOVERY_MAX_IMAGE_SNAPSHOT_CANDIDATES", MAX_IMAGE_SNAPSHOT_CANDIDATES)
)
# Keep review/inference/geocoding caps aligned with the visible candidate budget.
# These are source-backed limits, but the notebook should not expose them as
# independent knobs because doing so can leave discovered candidates unreviewed
# or without enrichment solely due to stale notebook defaults.
MAX_LLM_LOCATION_INFERENCES = str(_SOURCE_TOTAL_CANDIDATE_CAP)
MAX_CANDIDATE_REVIEWS = str(_SOURCE_TOTAL_CANDIDATE_CAP)
MAX_CANDIDATE_GEOCODES = str(_SOURCE_TOTAL_CANDIDATE_CAP)
MAX_STATE_SCALE_CANDIDATE_GEOCODES = str(_SOURCE_TOTAL_CANDIDATE_CAP)
# os.environ.pop("CAMERA_DISCOVERY_MAX_STREAMS", None)

# Preserve independent model choices. Do not normalize all stages to one model.
os.environ["CAMERA_DISCOVERY_LLM_PROVIDER"] = LLM_PROVIDER
os.environ["CAMERA_DISCOVERY_LLM_MODEL"] = MAIN_MODEL
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_MODEL"] = TARGET_INTENT_MODEL
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_FALLBACK_MODEL"] = TARGET_INTENT_FALLBACK_MODEL
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_TIMEOUT"] = TARGET_INTENT_TIMEOUT
os.environ["CAMERA_DISCOVERY_TARGET_INTENT_ATTEMPTS"] = TARGET_INTENT_ATTEMPTS
os.environ["CAMERA_DISCOVERY_GEOCODER_REFEREE_MODEL"] = GEOCODER_REFEREE_MODEL
os.environ["CAMERA_DISCOVERY_LOCATION_INFERENCE_MODEL"] = LOCATION_INFERENCE_MODEL
os.environ["CAMERA_DISCOVERY_LOCATION_INFERENCE_TIMEOUT"] = LOCATION_INFERENCE_TIMEOUT
os.environ["CAMERA_DISCOVERY_ENABLE_LLM_LOCATION_INFERENCE"] = ENABLE_LLM_LOCATION_INFERENCE
os.environ["CAMERA_DISCOVERY_MAX_LLM_LOCATION_INFERENCES"] = MAX_LLM_LOCATION_INFERENCES
os.environ["CAMERA_DISCOVERY_LOCATION_INFERENCE_MIN_CONFIDENCE"] = LOCATION_INFERENCE_MIN_CONFIDENCE
os.environ["CAMERA_DISCOVERY_CANDIDATE_REVIEW_MODEL"] = CANDIDATE_REVIEW_MODEL
os.environ["CAMERA_DISCOVERY_CANDIDATE_REVIEW_TIMEOUT"] = CANDIDATE_REVIEW_TIMEOUT
os.environ["CAMERA_DISCOVERY_CANDIDATE_REVIEW_BATCH_SIZE"] = CANDIDATE_REVIEW_BATCH_SIZE
os.environ["CAMERA_DISCOVERY_MAX_CANDIDATE_REVIEWS"] = MAX_CANDIDATE_REVIEWS
os.environ["CAMERA_DISCOVERY_MAX_SEARCH_QUERIES"] = MAX_SEARCH_QUERIES
os.environ["CAMERA_DISCOVERY_MAX_SEARCH_RESULTS_PER_QUERY"] = MAX_SEARCH_RESULTS_PER_QUERY
os.environ["CAMERA_DISCOVERY_MAX_PAGES"] = MAX_PAGES
os.environ["CAMERA_DISCOVERY_MAX_HLS_CANDIDATES"] = MAX_HLS_CANDIDATES
os.environ["CAMERA_DISCOVERY_MAX_IMAGE_SNAPSHOT_CANDIDATES"] = MAX_IMAGE_SNAPSHOT_CANDIDATES
os.environ[_TOTAL_CANDIDATE_CAP_ENV] = str(_SOURCE_TOTAL_CANDIDATE_CAP)
os.environ["CAMERA_DISCOVERY_MAX_DIRECTORY_PAGES"] = MAX_DIRECTORY_PAGES
os.environ["CAMERA_DISCOVERY_MAX_STRUCTURED_ENDPOINTS_PER_PAGE"] = MAX_STRUCTURED_ENDPOINTS_PER_PAGE
os.environ["CAMERA_DISCOVERY_MAX_CANDIDATE_GEOCODES"] = MAX_CANDIDATE_GEOCODES
os.environ["CAMERA_DISCOVERY_MAX_STATE_SCALE_CANDIDATE_GEOCODES"] = MAX_STATE_SCALE_CANDIDATE_GEOCODES
os.environ["CAMERA_DISCOVERY_IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS"] = IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS
os.environ["CAMERA_DISCOVERY_ENABLE_BROWSER_CAPTURE"] = ENABLE_BROWSER_CAPTURE
os.environ["CAMERA_DISCOVERY_BROWSER_BACKEND"] = BROWSER_BACKEND
os.environ["CAMERA_DISCOVERY_BROWSER_CAPTURE_TIMEOUT_MS"] = BROWSER_CAPTURE_TIMEOUT_MS
os.environ["CAMERA_DISCOVERY_BROWSER_CAPTURE_MIN_SCORE"] = BROWSER_CAPTURE_MIN_SCORE
os.environ["CAMERA_DISCOVERY_MAX_BROWSER_CAPTURE_PAGES"] = MAX_BROWSER_CAPTURE_PAGES
os.environ["CAMERA_DISCOVERY_MAX_BROWSER_CAPTURE_PAGES_BLIND"] = MAX_BROWSER_CAPTURE_PAGES_BLIND
os.environ["CAMERA_DISCOVERY_MAX_BROWSER_CAPTURE_PAGES_DIRECTORY"] = MAX_BROWSER_CAPTURE_PAGES_DIRECTORY
os.environ["CAMERA_DISCOVERY_MAX_BROWSER_CAPTURE_PAGES_PER_HOST"] = MAX_BROWSER_CAPTURE_PAGES_PER_HOST
os.environ["CAMERA_DISCOVERY_BROWSER_CAPTURE_SETTLE_MS"] = BROWSER_CAPTURE_SETTLE_MS
os.environ["CAMERA_DISCOVERY_BROWSER_CAPTURE_SCROLL"] = BROWSER_CAPTURE_SCROLL
os.environ["CAMERA_DISCOVERY_MAX_BROWSER_JSON_ENDPOINTS_PER_PAGE"] = MAX_BROWSER_JSON_ENDPOINTS_PER_PAGE
os.environ["CAMERA_DISCOVERY_MAX_BROWSER_NETWORK_EVENTS_LOGGED_PER_PAGE"] = MAX_BROWSER_NETWORK_EVENTS_LOGGED_PER_PAGE

DISCOVERY_MODE = os.environ.get("CAMERA_DISCOVERY_DISCOVERY_MODE", "both").strip().lower()
if DISCOVERY_MODE not in {"blind", "directory", "both", "direct"}:
    raise ValueError(f"Invalid CAMERA_DISCOVERY_DISCOVERY_MODE={DISCOVERY_MODE!r}")

SOURCES_FILE = Path(os.environ.get("CAMERA_DISCOVERY_SOURCES_FILE", "SOURCES.md"))
SEED_URLS = [url.strip() for url in os.environ.get("CAMERA_DISCOVERY_SEED_URLS", "").split(",") if url.strip()]

required_secret_hint = {
    "ollama": "OLLAMA_API_KEY is required only when using Ollama Cloud; local Ollama may not need it.",
    "ollama-cloud": "OLLAMA_API_KEY is required.",
    "ollama_cloud": "OLLAMA_API_KEY is required.",
    "openai-compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
    "openai_compatible": "OPENAI_API_KEY and OPENAI_BASE_URL are usually required.",
    "bedrock": "AWS credentials and AWS_DEFAULT_REGION are required.",
}.get(LLM_PROVIDER.lower(), "provider-specific credentials are required")

print("profile:", RUN_PROFILE)
# print("query:", USER_QUERY)
print("output:", OUTPUT_DIR)
print("clean output dir before run:", CLEAN_OUTPUT_DIR)
print("provider:", LLM_PROVIDER)
print("main model:", MAIN_MODEL)
print("target intent model:", TARGET_INTENT_MODEL)
print("target intent fallback model:", TARGET_INTENT_FALLBACK_MODEL)
print("target intent timeout:", TARGET_INTENT_TIMEOUT)
print("target intent attempts:", TARGET_INTENT_ATTEMPTS)
print("geocoder referee model:", GEOCODER_REFEREE_MODEL)
print("location inference model:", LOCATION_INFERENCE_MODEL)
print("location inference timeout:", LOCATION_INFERENCE_TIMEOUT)
print("enable LLM location inference:", ENABLE_LLM_LOCATION_INFERENCE)
print("max LLM location inferences (derived from HLS + image snapshot candidates):", MAX_LLM_LOCATION_INFERENCES)
print("location inference min confidence:", LOCATION_INFERENCE_MIN_CONFIDENCE)
print("candidate review model:", CANDIDATE_REVIEW_MODEL)
print("candidate review timeout:", CANDIDATE_REVIEW_TIMEOUT)
print("candidate review batch size:", CANDIDATE_REVIEW_BATCH_SIZE)
print("max candidate reviews (derived from HLS + image snapshot candidates):", MAX_CANDIDATE_REVIEWS)
print("max search queries:", MAX_SEARCH_QUERIES)
print("max search results per query:", MAX_SEARCH_RESULTS_PER_QUERY)
print("max pages:", MAX_PAGES)
print("max hls candidates:", MAX_HLS_CANDIDATES)
print("max image snapshot candidates:", MAX_IMAGE_SNAPSHOT_CANDIDATES)
print("max directory pages:", MAX_DIRECTORY_PAGES)
print("max structured endpoints per page:", MAX_STRUCTURED_ENDPOINTS_PER_PAGE)
print("browser capture enabled:", ENABLE_BROWSER_CAPTURE)
print("browser backend:", BROWSER_BACKEND)
print("browser capture timeout ms:", BROWSER_CAPTURE_TIMEOUT_MS)
print("browser capture min score:", BROWSER_CAPTURE_MIN_SCORE)
print("max browser pages total/blind/directory/host:", MAX_BROWSER_CAPTURE_PAGES, MAX_BROWSER_CAPTURE_PAGES_BLIND, MAX_BROWSER_CAPTURE_PAGES_DIRECTORY, MAX_BROWSER_CAPTURE_PAGES_PER_HOST)
print("browser capture settle ms / scroll:", BROWSER_CAPTURE_SETTLE_MS, BROWSER_CAPTURE_SCROLL)
print("browser JSON endpoints per page:", MAX_BROWSER_JSON_ENDPOINTS_PER_PAGE)
print("browser network events logged per page:", MAX_BROWSER_NETWORK_EVENTS_LOGGED_PER_PAGE)
print("max candidate geocodes (derived from HLS + image snapshot candidates):", MAX_CANDIDATE_GEOCODES)
print("max state-scale candidate geocodes (derived from HLS + image snapshot candidates):", MAX_STATE_SCALE_CANDIDATE_GEOCODES)
print("image snapshot refresh validation delay seconds:", IMAGE_SNAPSHOT_REFRESH_DELAY_SECONDS)
print("discovery mode:", DISCOVERY_MODE)
print("sources file:", SOURCES_FILE)
print("seed urls:", len(SEED_URLS))
print("credential hint:", required_secret_hint)

if DISCOVERY_MODE == "direct" and not SEED_URLS:
    raise ValueError("direct discovery mode requires CAMERA_DISCOVERY_SEED_URLS or --seed-url values")

if LLM_PROVIDER.lower() in {"ollama-cloud", "ollama_cloud"} and not os.environ.get("OLLAMA_API_KEY"):
    print("WARNING: OLLAMA_API_KEY is not set; Ollama Cloud requests will fail until configured.")

In [ ]:
!camera-discovery run "California traffic cameras" \
  --profile "$RUN_PROFILE" \
  --output-dir runs/run-from-harvest-hls \
  --harvest-input runs/harvest-california-hls/harvest_handoff.json \
  --harvest-input-mode handoff-only \
  --browser-backend cloakbrowser \
  --progress-style plain


## Optional: zip and download harvest outputs


In [ ]:
from pathlib import Path
import shutil

out_dir = Path('runs/harvest-california')
if out_dir.exists():
    archive = shutil.make_archive(str(out_dir), 'zip', root_dir=out_dir)
    print('Created:', archive)
    try:
        from google.colab import files
        files.download(archive)
    except Exception as exc:
        print('Download helper unavailable outside Colab:', exc)
else:
    print('Run harvest first; output directory does not exist:', out_dir)


## Timestamp promotion and image asset filtering checks

The cells below inspect harvest outputs only. They do not patch source files or implement extraction logic.


In [ ]:
from pathlib import Path
import json

HARVEST_DIR = Path('runs/harvest-california-hls')
for name in ['camera_records.jsonl', 'camera_urls.jsonl', 'harvest_camera_inventory.jsonl']:
    path = HARVEST_DIR / name
    print(name, 'exists=', path.exists())
    if path.exists():
        shown = 0
        for line in path.read_text(encoding='utf-8').splitlines():
            row = json.loads(line)
            if any(row.get(k) for k in ['date', 'time', 'timestamp', 'last_updated', 'last_refresh']):
                print(json.dumps({
                    'camera_record_id': row.get('camera_record_id'),
                    'url': row.get('url'),
                    'date': row.get('date'),
                    'time': row.get('time'),
                    'timestamp': row.get('timestamp'),
                    'last_updated': row.get('last_updated'),
                    'last_refresh': row.get('last_refresh'),
                    'field_map': {k: v for k, v in (row.get('field_map') or {}).items() if 'date' in k.lower() or 'time' in k.lower() or 'epoch' in k.lower()},
                }, indent=2)[:4000])
                shown += 1
                break
        if shown == 0:
            print('No promoted temporal fields found in', name)


## Optional stricter image filtering

Default all-media harvest is intentionally broad and may include page assets. Use `--image-asset-filter camera-evidence` when you want image URLs with stronger camera/snapshot evidence.


In [ ]:
!camera-discovery harvest-urls "California traffic cameras"   --output-dir runs/harvest-california-images-camera-evidence   --max-urls 10000   --media image   --image-asset-filter camera-evidence   --disable-browser-capture


In [ ]:
from pathlib import Path
import json

SUMMARY = Path('runs/harvest-california-images-camera-evidence/harvest_summary.json')
if SUMMARY.exists():
    summary = json.loads(SUMMARY.read_text(encoding='utf-8'))
    for key in [
        'image_asset_filter',
        'image_asset_filter_removed',
        'image_asset_filter_kept',
        'image_asset_filter_removed_by_reason',
        'pre_image_filter_records',
        'post_image_filter_records',
        'records_with_datetime',
        'records_with_update_frequency',
    ]:
        print(f'{key}:', summary.get(key))
else:
    print('Run the image-filter harvest cell first.')


## Intermediate debug files can be large

`--write-intermediate-records` is opt-in because raw/unique JSONL debug files can be very large.


In [ ]:
from pathlib import Path

HARVEST_DIR = Path('runs/harvest-california-hls')
for name in ['raw_media_records.jsonl', 'unique_media_records.jsonl', 'media_filtered_records.jsonl', 'image_filtered_records.jsonl']:
    path = HARVEST_DIR / name
    if path.exists():
        print(f'{name}: {path.stat().st_size / (1024 * 1024):.2f} MB')
    else:
        print(f'{name}: not written; rerun with --write-intermediate-records')


## Handoff run credential preflight

Harvest mode is extraction-only and unvalidated. Handoff input is source-provided but not trusted. Normal `run` mode still performs target resolution, scope, validation/trust/output behavior according to configuration.


In [ ]:
import os
import subprocess
from pathlib import Path

provider = os.getenv('CAMERA_DISCOVERY_LLM_PROVIDER', 'ollama-cloud').strip().lower()
missing = []
if provider in {'ollama', 'ollama-cloud'} and not os.getenv('OLLAMA_API_KEY') and provider == 'ollama-cloud':
    missing.append('OLLAMA_API_KEY')
elif provider in {'openai', 'openai-compatible', 'openai_compatible'} and not os.getenv('OPENAI_COMPATIBLE_API_KEY'):
    missing.append('OPENAI_COMPATIBLE_API_KEY')
elif provider == 'bedrock' and not (os.getenv('AWS_PROFILE') or os.getenv('AWS_ACCESS_KEY_ID')):
    missing.append('AWS_PROFILE or AWS_ACCESS_KEY_ID')

handoff = Path('runs/harvest-california-hls/harvest_handoff.json')
if missing:
    print(f"Skipping run --harvest-input demonstration because {', '.join(missing)} is not set for provider {provider}.")
    print('Harvest handoff files were created successfully. Configure credentials and rerun this cell to test normal run mode.')
elif not handoff.exists():
    print('Skipping run --harvest-input demonstration because the handoff file is missing:', handoff)
else:
    manifest = json.loads(handoff.read_text(encoding='utf-8'))
    print('Harvest handoff schema:', manifest.get('schema_version'))
    print('Harvest handoff default scope:', manifest.get('handoff_default_scope'))
    print('Harvest handoff media filter:', manifest.get('media_filter'))
    print('Harvest handoff counts:', manifest.get('counts'))
    print('Expected handoff-only behavior: normal discovery disabled; candidate count should be bounded by selected handoff records and target count.')
    print('Use --harvest-input-mode seed only when intentionally combining the handoff with normal discovery.')
    cmd = [
        'camera-discovery', 'run', 'California traffic cameras',
        '--profile', RUN_PROFILE,
        '--output-dir', 'runs/run-from-harvest',
        '--harvest-input', str(handoff),
        '--harvest-input-mode', 'handoff-only',
        '--browser-backend', 'cloakbrowser',
        '--progress-style', 'plain',
    ]
    print('Running:', ' '.join(cmd))
    subprocess.run(cmd, check=True)
